# 05 · Systematic evaluation

**Purpose.** Evaluate every system variant on the same queries and snapshots:
1. **Ranking quality:** nDCG@5 (primary), MRR, P@3, Recall@10, per query and averaged, with paired significance tests and effect sizes.
2. **Surfability calls vs. buoy observations:** accuracy, precision/recall, confusion matrix, forecast error, and how forecast error relates to ranking quality.
3. **Efficiency:** indexing time and size, median/p95 query latency, snapshot load vs. live API time.

**Inputs:** `eval/queries.tsv` + `eval/qrels.txt` (TREC format; see `eval/README.md`), snapshots in `data/snapshots/`, the search functions from `04_search` (loaded with `%run`, not copied).

**Outputs** (`results/`): `summary.csv`, `per_query.csv`, `significance.csv`, `by_query_type.csv`, `surfability_metrics.csv`, `forecast_error.csv`, `latency.csv`, `judging_pool.csv`, `runs/*.run` (TREC run files), `tables/*.md` (report-ready), `figures/*.png`.

> ⚠️ By default this notebook uses the **smoke-test** files with **mock judgments**, only to prove the pipeline runs. Switch `QUERIES_FILE` / `QRELS_FILE` to your real collection before reporting anything.

In [ ]:
# ---- Configuration ----
import os
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
os.chdir(ROOT / "notebooks")
os.environ["SURF_IR_LIBRARY_MODE"] = "1"     # load 04 (and through it 02 and 03) as a library: no example output
%run ./04_search.ipynb
# Everything below is set AFTER %run, so these values win over any same-named settings in 02-04.

EVAL_DIR = ROOT / "eval"
QUERIES_FILE = EVAL_DIR / "smoke_test_queries.tsv"     # real collection: EVAL_DIR / "queries.tsv"
QRELS_FILE = EVAL_DIR / "smoke_test_qrels.txt"         # real collection: EVAL_DIR / "qrels.txt"
QRELS_SECOND_JUDGE_FILE = EVAL_DIR / "qrels_judge2.txt"  # optional, for Cohen's kappa
RESULTS_DIR = ROOT / "results"
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
RUN_DIR = RESULTS_DIR / "runs"

EVAL_SPLIT = "all"             # "test" for reported results, "dev" while developing, "all" for the smoke test
SAMPLE_N = None                # e.g. 10 = only the first 10 queries (quick development runs)
SEED = 42
BASELINE_SYSTEM = "baseline"
MAIN_SYSTEM = "improved"
RUN_DEPTH = 10                 # results retrieved per query (= pool depth for judging)
REL_THRESHOLD = 2              # grade >= 2 counts as relevant for binary metrics (MRR, P@3, Recall@10)
PRIMARY_METRIC = "ndcg@5"
N_PERMUTATIONS = 10_000        # paired randomisation test
N_BOOTSTRAP = 10_000           # bootstrap confidence intervals
SIG_LEVEL = 0.05
TUNE_ALPHA = False             # True = choose ALPHA on the dev split (grid below), then evaluate
ALPHA_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
SURFABILITY_SKILL = "intermediate"
MAX_LEAD_DAYS = 2              # compare forecasts made 0, 1, 2 days ahead with observations
MIN_OBS_HOURS = 6              # a day needs at least this many observed daylight hours to be judged
LATENCY_REPEATS = 5
MEASURE_LIVE_API = False       # True = also time one live API collection (needs internet; ~2 requests per spot)

## 1. Load the test collection
Each query carries its own **snapshot date**: relevance depends on the conditions, so a judgment is only valid for the snapshot it was made on. The TREC formats let you cross-check our metric code with standard tools (`trec_eval`, `ir_measures`) if a reviewer asks.

In [ ]:
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats
from sklearn.metrics import cohen_kappa_score, confusion_matrix

rng = np.random.default_rng(SEED)
for d in (RESULTS_DIR, FIG_DIR, TABLE_DIR, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)


def load_queries(path=QUERIES_FILE, split=EVAL_SPLIT, sample_n=SAMPLE_N):
    q = pd.read_csv(path, sep="\t", comment="#", dtype=str).fillna("")
    q["snapshot_date"] = q["snapshot_date"].replace({"": "latest"}).apply(resolve_snapshot_date)
    if split != "all":
        q = q[q["split"] == split]
    return (q.head(sample_n) if sample_n else q).reset_index(drop=True)


def load_qrels(path=QRELS_FILE):
    rows = [line.split() for line in path.read_text(encoding="utf-8").splitlines() if line.strip() and not line.startswith("#")]
    qrels = {}
    for qid, _, doc, grade in rows:
        qrels.setdefault(qid, {})[doc] = int(grade)
    return qrels


queries = load_queries()
qrels = load_qrels()
if "smoke_test" in QRELS_FILE.name:
    display(Markdown("> ⚠️ **Smoke test:** mock judgments. These numbers mean nothing; do not report them."))
missing = sorted(set(queries["qid"]) - set(qrels))
print(f"{len(queries)} queries ({EVAL_SPLIT} split), {sum(len(v) for v in qrels.values())} judgments; "
      f"snapshots used: {sorted(queries['snapshot_date'].unique())}")
if missing:
    print("Queries without any judgment (excluded):", missing)
    queries = queries[~queries["qid"].isin(missing)].reset_index(drop=True)
display(queries)

## 2. Metrics
- **nDCG@5 (primary):** users compare only a handful of spots, and our judgments are graded (0–3). nDCG rewards putting the *best-fitting* spot first, not just any relevant one. Gain = grade, discount = log₂(rank + 1), as in `trec_eval`.
- **MRR:** "where should I go?" is often answered by the first good spot.
- **P@3:** the first screen on a phone.
- **Recall@10:** coverage for exploratory queries ("best waves this weekend").
- **judged@5:** share of the top 5 that has a judgment. A low value means the pool was too shallow and scores are underestimated (a diagnostic, not a quality metric).

In [ ]:
def dcg(grades):
    return sum(g / np.log2(i + 2) for i, g in enumerate(grades))


def evaluate_ranking(ranked, judged, k_ndcg=5, k_p=3, k_r=10, thr=REL_THRESHOLD):
    grades = [judged.get(d, 0) for d in ranked]
    ideal = sorted(judged.values(), reverse=True)
    n_rel = sum(g >= thr for g in judged.values())
    first_rel = next((i for i, g in enumerate(grades[:10]) if g >= thr), None)
    return {
        "ndcg@5": dcg(grades[:k_ndcg]) / dcg(ideal[:k_ndcg]) if dcg(ideal[:k_ndcg]) > 0 else 0.0,
        "mrr": 0.0 if first_rel is None else 1.0 / (first_rel + 1),
        "p@3": sum(g >= thr for g in grades[:k_p]) / k_p,
        "recall@10": sum(g >= thr for g in grades[:k_r]) / n_rel if n_rel else 0.0,
        "judged@5": sum(d in judged for d in ranked[:5]) / max(min(5, len(ranked)), 1),
    }


METRICS = ["ndcg@5", "mrr", "p@3", "recall@10"]

# sanity checks on hand-computed examples
_j = {"a": 3, "b": 2, "c": 0}
assert abs(evaluate_ranking(["a", "b", "c"], _j)["ndcg@5"] - 1.0) < 1e-9
assert abs(evaluate_ranking(["c", "a"], _j)["mrr"] - 0.5) < 1e-9
assert abs(evaluate_ranking(["b", "x", "a"], _j)["p@3"] - 2 / 3) < 1e-9
print("Metric sanity checks passed.")

## 3. Run every system
All systems see exactly the same queries, snapshots and reference times. Runs are also written as TREC run files, the input format for pooling and for external evaluation tools.

In [ ]:
HAS_OBSERVATIONS = not load_all_observations().empty
SYSTEMS_TO_EVAL = [s for s in AVAILABLE_SYSTEMS if s != "improved_oracle" or HAS_OBSERVATIONS]
if not HAS_OBSERVATIONS:
    print("No CWA observations in any snapshot: 'improved_oracle' and the surfability checks are skipped.")
print("Systems:", SYSTEMS_TO_EVAL)


def run_system(system, qs, alpha=None):
    rows = []
    for q in qs.itertuples():
        kwargs = {"skill": getattr(q, "skill", "") or None, "ref_time": getattr(q, "ref_time", "") or None}
        results = search(q.query, system=system, k=RUN_DEPTH, snapshot_date=q.snapshot_date, alpha=alpha, **kwargs)
        rows += [{"qid": q.qid, "system": system, "rank": r["rank"], "spot_id": r["spot_id"], "score": r["score"]}
                 for r in results]
    return pd.DataFrame(rows)


def per_query_metrics(run, qs):
    rows = []
    for q in qs.itertuples():
        ranked = run.loc[run["qid"] == q.qid].sort_values("rank")["spot_id"].tolist()
        rows.append({"qid": q.qid, "query_type": q.query_type, **evaluate_ranking(ranked, qrels.get(q.qid, {}))})
    return pd.DataFrame(rows)


def write_trec_run(run, system):
    with open(RUN_DIR / f"{system}.run", "w", encoding="utf-8") as f:
        for r in run.itertuples():
            f.write(f"{r.qid} Q0 {r.spot_id} {r.rank} {r.score:.6f} {system}\n")

### Optional: tune α on the dev queries
α balances text relevance and condition fit. Choosing it on the *test* queries would overfit the very numbers we report, so it is chosen on the dev split only and then frozen.

In [ ]:
TUNED_ALPHA = None
if TUNE_ALPHA:
    dev = load_queries(split="dev")
    scores = {a: per_query_metrics(run_system(MAIN_SYSTEM, dev, alpha=a), dev)[PRIMARY_METRIC].mean() for a in ALPHA_GRID}
    TUNED_ALPHA = max(scores, key=scores.get)
    plt.figure(figsize=(5, 3))
    plt.plot(list(scores), list(scores.values()), marker="o")
    plt.xlabel("alpha (weight of text relevance)")
    plt.ylabel(f"mean {PRIMARY_METRIC} (dev)")
    plt.title(f"Alpha tuning on dev: best = {TUNED_ALPHA}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "alpha_tuning.png", dpi=200)
    plt.show()
print("Alpha used:", TUNED_ALPHA if TUNED_ALPHA is not None else f"default (ALPHA={ALPHA}, no-condition-intent={ALPHA_NO_CONDITION_INTENT})")

In [ ]:
runs, per_query = [], []
for system in SYSTEMS_TO_EVAL:
    run = run_system(system, queries, alpha=TUNED_ALPHA if system != BASELINE_SYSTEM else None)
    write_trec_run(run, system)
    runs.append(run)
    per_query.append(per_query_metrics(run, queries).assign(system=system))
runs = pd.concat(runs, ignore_index=True)
per_query = pd.concat(per_query, ignore_index=True)
per_query.to_csv(RESULTS_DIR / "per_query.csv", index=False)
print(f"{len(runs)} result rows written to {RUN_DIR.relative_to(ROOT)}/")

## 4. Summary with confidence intervals
Means alone hide how uncertain they are with ~35 test queries, so every mean comes with a 95% bootstrap confidence interval (resampling queries).

In [ ]:
def bootstrap_ci(values, n=N_BOOTSTRAP, level=0.95):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return (np.nan, np.nan)
    means = rng.choice(values, size=(n, len(values)), replace=True).mean(axis=1)
    return tuple(np.quantile(means, [(1 - level) / 2, 1 - (1 - level) / 2]))


summary_rows = []
for system, g in per_query.groupby("system", sort=False):
    row = {"system": system, "n_queries": len(g)}
    for m in METRICS + ["judged@5"]:
        row[m] = g[m].mean()
    lo, hi = bootstrap_ci(g[PRIMARY_METRIC])
    row[f"{PRIMARY_METRIC}_ci_low"], row[f"{PRIMARY_METRIC}_ci_high"] = lo, hi
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
display(summary.round(3))

## 5. Paired significance tests and effect sizes
Every system answers the *same* queries, so differences are tested **per query (paired)**:
- **Randomisation (permutation) test:** randomly flips the sign of each per-query difference to see how often a mean difference this large appears by chance. It makes no normality assumption; Smucker et al. (2007) recommend it for IR evaluation.
- **Paired t-test:** reported alongside as the familiar reference.
- **Holm–Bonferroni correction:** we make several comparisons, so p-values are adjusted for multiple testing.
- **Effect sizes:** mean difference with a 95% bootstrap CI, and Cohen's d_z (mean difference / SD of differences). With few queries a real improvement can be non-significant; the effect size shows its magnitude anyway.

Comparisons: improved vs. baseline (the main hypothesis), and each ablation vs. improved (what each component contributes).

In [ ]:
def randomization_test(diffs, n=N_PERMUTATIONS):
    diffs = np.asarray(diffs, dtype=float)
    observed = abs(diffs.mean())
    signs = rng.choice([-1.0, 1.0], size=(n, len(diffs)))
    return (np.sum(np.abs((signs * diffs).mean(axis=1)) >= observed - 1e-12) + 1) / (n + 1)


def holm(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, min(1.0, (len(p) - rank) * p[idx]))
        adjusted[idx] = running
    return adjusted


comparisons = [(MAIN_SYSTEM, BASELINE_SYSTEM)] + [(s, MAIN_SYSTEM) for s in SYSTEMS_TO_EVAL if s not in (MAIN_SYSTEM, BASELINE_SYSTEM)]
sig_rows = []
wide = per_query.pivot(index="qid", columns="system", values=PRIMARY_METRIC)
for a, b in comparisons:
    d = (wide[a] - wide[b]).values
    sd = d.std(ddof=1) if len(d) > 1 else np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t_p = stats.ttest_rel(wide[a], wide[b]).pvalue if len(d) > 1 and sd > 0 else np.nan
    lo, hi = bootstrap_ci(d)
    sig_rows.append({"system": a, "vs": b, "metric": PRIMARY_METRIC, "mean_a": wide[a].mean(), "mean_b": wide[b].mean(),
                     "mean_diff": d.mean(), "diff_ci_low": lo, "diff_ci_high": hi,
                     "cohens_dz": d.mean() / sd if sd and sd > 0 else np.nan,
                     "wins": int((d > 0).sum()), "ties": int((d == 0).sum()), "losses": int((d < 0).sum()),
                     "p_randomization": randomization_test(d), "p_ttest": t_p})
significance = pd.DataFrame(sig_rows)
significance["p_randomization_holm"] = holm(significance["p_randomization"])
significance["significant"] = significance["p_randomization_holm"] < SIG_LEVEL
significance.to_csv(RESULTS_DIR / "significance.csv", index=False)
if len(queries) < 20:
    print(f"Only {len(queries)} queries: p-values are not meaningful at this size.")
display(significance.round(4))

## 6. Where does the improved system win or lose?
Averages hide the story. The per-query difference plot shows *which* queries gain and which lose, coloured by query type. Use it to pick queries for the failure analysis: for every loss, open the run and assign a failure category (query-understanding miss, vocabulary gap, threshold wrong, forecast error, coarse grid / sheltered spot, judging ambiguity). **[TEAM]**

In [ ]:
delta = (wide[MAIN_SYSTEM] - wide[BASELINE_SYSTEM]).rename("delta").reset_index()
delta = delta.merge(queries[["qid", "query", "query_type"]], on="qid").sort_values("delta")
types = sorted(delta["query_type"].unique())
colors = dict(zip(types, plt.cm.tab10.colors))
fig, ax = plt.subplots(figsize=(max(6, 0.35 * len(delta) + 2), 3.8))
ax.bar(range(len(delta)), delta["delta"], color=[colors[t] for t in delta["query_type"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(delta)), delta["qid"], rotation=90)
ax.set_ylabel(f"Δ {PRIMARY_METRIC} (improved − baseline)")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=colors[t]) for t in types], labels=types, fontsize=8, ncol=3)
ax.set_title("Per-query difference, sorted")
fig.tight_layout()
fig.savefig(FIG_DIR / "per_query_delta.png", dpi=200)
plt.show()
display(delta.round(3))

In [ ]:
by_type = per_query.groupby(["query_type", "system"])[PRIMARY_METRIC].mean().unstack("system")[SYSTEMS_TO_EVAL]
by_type["n"] = queries["query_type"].value_counts()
by_type.to_csv(RESULTS_DIR / "by_query_type.csv")
ax = by_type[SYSTEMS_TO_EVAL].plot.bar(figsize=(max(6, 1.2 * len(by_type) + 3), 3.8), width=0.8)
ax.set_ylabel(f"mean {PRIMARY_METRIC}")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.legend(fontsize=7, ncol=2)
ax.set_title("Effectiveness by query type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "by_query_type.png", dpi=200)
plt.show()
display(by_type.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(1.1 * len(summary) + 2, 3.6))
err = np.vstack([summary[PRIMARY_METRIC] - summary[f"{PRIMARY_METRIC}_ci_low"],
                 summary[f"{PRIMARY_METRIC}_ci_high"] - summary[PRIMARY_METRIC]])
ax.bar(summary["system"], summary[PRIMARY_METRIC], yerr=np.nan_to_num(err), capsize=4,
       color=["#9E9E9E" if s == BASELINE_SYSTEM else "#4C78A8" for s in summary["system"]])
ax.set_ylabel(f"mean {PRIMARY_METRIC} (95% CI)")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right")
ax.set_title("Systems and ablations")
fig.tight_layout()
fig.savefig(FIG_DIR / "summary_primary_metric.png", dpi=200)
plt.show()

## 7. Judging pool and inter-annotator agreement
**Pool:** the union of the top `RUN_DEPTH` results of all systems per query. This is the list your judges must label. Re-export it whenever a system changes, and judge any new (unjudged) spots.

**Agreement:** if a second judge labelled an overlap sample (`qrels_judge2.txt`), we report Cohen's kappa: quadratic-weighted for the graded scale (a 3-vs-2 disagreement is milder than 3-vs-0), and unweighted on the binarised labels (relevant = grade ≥ 2).

In [ ]:
pool = (runs.groupby(["qid", "spot_id"])["system"].apply(lambda s: ";".join(sorted(set(s)))).reset_index())
pool["already_judged"] = [s in qrels.get(q, {}) for q, s in zip(pool["qid"], pool["spot_id"])]
pool = pool.merge(queries[["qid", "query", "snapshot_date"]], on="qid")
pool.to_csv(RESULTS_DIR / "judging_pool.csv", index=False)
print(f"Judging pool: {len(pool)} (query, spot) pairs, {int((~pool['already_judged']).sum())} not yet judged "
      f"-> {(RESULTS_DIR / 'judging_pool.csv').relative_to(ROOT)}")

In [ ]:
if QRELS_SECOND_JUDGE_FILE.exists():
    q2 = load_qrels(QRELS_SECOND_JUDGE_FILE)
    pairs = [(qrels[q][d], g) for q, docs in q2.items() for d, g in docs.items() if d in qrels.get(q, {})]
    a, b = np.array(pairs).T
    kappa = {"pairs": len(pairs), "kappa_quadratic": cohen_kappa_score(a, b, weights="quadratic"),
             "kappa_binary": cohen_kappa_score(a >= REL_THRESHOLD, b >= REL_THRESHOLD)}
    pd.Series(kappa).to_csv(RESULTS_DIR / "agreement.csv")
    display(pd.Series(kappa).round(3).to_frame("value"))
else:
    print(f"No second-judge file ({QRELS_SECOND_JUDGE_FILE.name}); agreement not computed.")

## 8. Surfability calls vs. CWA buoy observations
For each spot with a buoy, each snapshot day *D* and each lead time *L* (0, 1, 2 days), we compare:
- the **forecast call**: the condition-fit rules applied to the forecast made on day *D* for day *D + L*;
- the **observed call**: the *same* rules applied to what the buoy actually measured on day *D + L*.

Because both calls use identical rules, disagreements isolate **forecast error**. They do not test whether the rules themselves are right (that needs surfers' session reports; see Limitations). Remember too that buoys measure offshore conditions, not the beach.

In [ ]:
def surfability_comparison(skill=SURFABILITY_SKILL):
    obs_all = load_all_observations()
    t = THRESHOLDS[skill]
    calls, hourly = [], []
    if obs_all.empty:
        return pd.DataFrame(), pd.DataFrame()
    for snap_date in list_snapshots():
        snap = load_snapshot(snap_date)
        for sid, spot in SPOTS.items():
            obs_frame = observed_frame(sid, snap, obs_all)
            fc_frame = forecast_frame(sid, snap)
            if obs_frame.empty or fc_frame.empty:
                continue
            for lead in range(MAX_LEAD_DAYS + 1):
                day = pd.Timestamp(snap_date) + pd.Timedelta(days=lead)
                sel = lambda f: f[(f.index.normalize() == day) & (f.index.hour >= t["first_light_hour"]) & (f.index.hour < t["last_light_hour"])]
                fc_day, ob_day = sel(fc_frame), sel(obs_frame)
                if len(ob_day) < MIN_OBS_HOURS or fc_day.empty:
                    continue
                fc_cf = factor_scores(fc_day, spot, skill)["cf"].max()
                ob_cf = factor_scores(ob_day, spot, skill)["cf"].max()
                calls.append({"spot_id": sid, "snapshot": snap_date, "day": day.date(), "lead_days": lead,
                              "forecast_cf": fc_cf, "observed_cf": ob_cf,
                              "forecast_surfable": fc_cf >= t["surfable_threshold"], "observed_surfable": ob_cf >= t["surfable_threshold"]})
                joined = fc_day[["wave_height_m"]].join(ob_day[["wave_height_m"]], lsuffix="_fc", rsuffix="_obs", how="inner")
                hourly.append(joined.assign(spot_id=sid, lead_days=lead, station=buoy_for_spot(spot)[1]))
    return pd.DataFrame(calls), (pd.concat(hourly).reset_index() if hourly else pd.DataFrame())


surf_calls, hs_pairs = surfability_comparison()
if surf_calls.empty:
    print("No (spot, day) pairs with both forecast and buoy observations yet. Needs: CWA key, buoy mapping "
          "(buoy_station_id or station coordinates in cwa_stations_en.csv), and snapshots covering the days.")

In [ ]:
def call_metrics(g, lead):
    tp, fn, fp, tn = confusion_matrix(g["observed_surfable"], g["forecast_surfable"], labels=[True, False]).ravel()
    return {"lead_days": lead, "pairs": len(g), "accuracy": (tp + tn) / len(g),
            "precision_surfable": tp / (tp + fp) if tp + fp else np.nan,
            "recall_surfable": tp / (tp + fn) if tp + fn else np.nan,
            "observed_surfable_rate": g["observed_surfable"].mean()}


if not surf_calls.empty:
    cm = confusion_matrix(surf_calls["observed_surfable"], surf_calls["forecast_surfable"], labels=[True, False])
    surf_metrics = pd.DataFrame([call_metrics(surf_calls, "all")] +
                                [call_metrics(g, lead) for lead, g in surf_calls.groupby("lead_days")])
    surf_metrics.to_csv(RESULTS_DIR / "surfability_metrics.csv", index=False)
    display(surf_metrics.round(3))

    fig, ax = plt.subplots(figsize=(3.8, 3.4))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=12)
    ax.set_xticks([0, 1], ["surfable", "not surfable"])
    ax.set_yticks([0, 1], ["surfable", "not surfable"])
    ax.set_xlabel("forecast call")
    ax.set_ylabel("observed call (buoy)")
    ax.set_title(f"Surfability calls ({SURFABILITY_SKILL})")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "surfability_confusion_matrix.png", dpi=200)
    plt.show()

### Forecast error: wave height
MAE, bias (forecast − observed; positive = forecast too big) and RMSE per lead time and per buoy. A systematic bias at one buoy is a finding in itself, e.g. the model grid not resolving a sheltered coast.

In [ ]:
if not hs_pairs.empty:
    hs_pairs["error"] = hs_pairs["wave_height_m_fc"] - hs_pairs["wave_height_m_obs"]
    agg = lambda g: pd.Series({"hours": len(g), "mae_m": g["error"].abs().mean(), "bias_m": g["error"].mean(),
                               "rmse_m": np.sqrt((g["error"] ** 2).mean())})
    forecast_error = pd.concat([hs_pairs.groupby("lead_days").apply(agg, include_groups=False).reset_index().assign(group="lead"),
                                hs_pairs.groupby("station").apply(agg, include_groups=False).reset_index().assign(group="station")])
    forecast_error.to_csv(RESULTS_DIR / "forecast_error.csv", index=False)
    display(forecast_error.round(3))

    fig, ax = plt.subplots(figsize=(4.2, 4))
    ax.scatter(hs_pairs["wave_height_m_obs"], hs_pairs["wave_height_m_fc"], s=8, alpha=0.4)
    lim = [0, max(hs_pairs[["wave_height_m_obs", "wave_height_m_fc"]].max().max(), 0.5) * 1.05]
    ax.plot(lim, lim, color="black", linewidth=0.8)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel("observed wave height (m)")
    ax.set_ylabel("forecast wave height (m)")
    ax.set_title("Forecast vs buoy")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "forecast_vs_observed.png", dpi=200)
    plt.show()

### Does forecast error hurt ranking quality?
For each query we take the improved system's top-5 spots and compute the mean absolute wave-height error over the query's time window (where observations exist). We then relate it to the query's Δ nDCG@5 (improved − baseline). The `improved_oracle` system (conditions from observations) in the summary table gives the complementary view: how much better ranking would be with a perfect forecast.

In [ ]:
if not hs_pairs.empty:
    rows = []
    for q in queries.itertuples():
        info = search_verbose(q.query, MAIN_SYSTEM, k=5, snapshot_date=q.snapshot_date)
        windows = info["parsed"].windows
        top = [r["spot_id"] for r in info["results"]]
        sel = hs_pairs[hs_pairs["spot_id"].isin(top) & (hs_pairs["lead_days"] >= 0)]
        sel = sel[np.any([(sel["time"] >= s) & (sel["time"] < e) for s, e in windows], axis=0)] if windows else sel.iloc[0:0]
        if len(sel):
            rows.append({"qid": q.qid, "abs_error_m": sel["error"].abs().mean()})
    err_vs_delta = pd.DataFrame(rows, columns=["qid", "abs_error_m"]).merge(delta[["qid", "delta"]], on="qid")
    if len(err_vs_delta) >= 3:
        rho, p = stats.spearmanr(err_vs_delta["abs_error_m"], err_vs_delta["delta"])
        print(f"Spearman rho = {rho:.2f} (p = {p:.3f}, n = {len(err_vs_delta)})")
        plt.figure(figsize=(4.2, 3.6))
        plt.scatter(err_vs_delta["abs_error_m"], err_vs_delta["delta"])
        plt.axhline(0, color="black", linewidth=0.8)
        plt.xlabel("mean |forecast − observed| wave height (m)")
        plt.ylabel(f"Δ {PRIMARY_METRIC}")
        plt.title("Forecast error vs ranking gain")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "forecast_error_vs_delta.png", dpi=200)
        plt.show()
    else:
        print("Too few queries with observations in their time window for this analysis.")

## 9. Efficiency
Users expect an answer in well under a second. We measure end-to-end `search()` latency per system (repeated runs on a warm cache, as in the demo), plus how long loading a snapshot takes compared with fetching live data.

In [ ]:
lat_rows = []
latency_systems = [s for s in SYSTEMS_TO_EVAL if s != "improved_oracle"]
for system in latency_systems:
    search(queries["query"].iloc[0], system=system, snapshot_date=queries["snapshot_date"].iloc[0])   # warm-up
    for q in queries.itertuples():
        for _ in range(LATENCY_REPEATS):
            t0 = time.perf_counter()
            search(q.query, system=system, k=RUN_DEPTH, snapshot_date=q.snapshot_date)
            lat_rows.append({"system": system, "ms": (time.perf_counter() - t0) * 1000})
latency_raw = pd.DataFrame(lat_rows)
latency = latency_raw.groupby("system", sort=False)["ms"].agg(
    median_ms="median", p95_ms=lambda x: np.percentile(x, 95), mean_ms="mean", runs="count").reset_index()

load_times = []
for _ in range(3):
    _SNAPSHOT_CACHE.clear()
    _FRAME_CACHE.clear()
    t0 = time.perf_counter()
    load_snapshot(queries["snapshot_date"].iloc[0])
    load_times.append((time.perf_counter() - t0) * 1000)
extra = [{"system": "snapshot load (cold)", "median_ms": np.median(load_times), "p95_ms": np.max(load_times),
          "mean_ms": np.mean(load_times), "runs": len(load_times)}]
latency = pd.concat([latency, pd.DataFrame(extra)], ignore_index=True)
latency.to_csv(RESULTS_DIR / "latency.csv", index=False)
display(latency.round(1))

In [ ]:
if MEASURE_LIVE_API:
    os.environ["SURF_IR_LIBRARY_MODE"] = "1"
    %run ./00_data_collection.ipynb
    t0 = time.perf_counter()
    live = collect_all()
    live_s = time.perf_counter() - t0
    print(f"Live collection for {live['meta']['n_spots']} spots: {live_s:.1f} s (status {live['meta']['status']})")
    latency = pd.concat([latency, pd.DataFrame([{"system": "live API collection", "median_ms": live_s * 1000,
                                                 "p95_ms": live_s * 1000, "mean_ms": live_s * 1000, "runs": 1}])], ignore_index=True)
    latency.to_csv(RESULTS_DIR / "latency.csv", index=False)

In [ ]:
plt.figure(figsize=(1.2 * len(latency_systems) + 2, 3.4))
plt.boxplot([latency_raw.loc[latency_raw["system"] == s, "ms"] for s in latency_systems], showfliers=False)
plt.xticks(range(1, len(latency_systems) + 1), latency_systems, rotation=30, ha="right")
plt.ylabel("query latency (ms)")
plt.title("Query latency per system")
plt.tight_layout()
plt.savefig(FIG_DIR / "latency.png", dpi=200)
plt.show()
index_stats = json.loads((RESULTS_DIR / "indexing_stats.json").read_text(encoding="utf-8"))
display(pd.Series(index_stats, name="indexing").to_frame())

## 10. Report-ready tables
Markdown versions of the main tables (paste into the report, or convert with pandoc). Numbers are rounded to 3 decimals; the CSV files keep full precision.

In [ ]:
def save_table(df, name, caption):
    (TABLE_DIR / f"{name}.md").write_text(f"**{caption}**\n\n" + df.to_markdown(index=False, floatfmt=".3f") + "\n", encoding="utf-8")


main_table = summary[["system", "n_queries"] + METRICS].copy()
main_table[f"{PRIMARY_METRIC} 95% CI"] = [f"[{lo:.3f}, {hi:.3f}]" for lo, hi in
                                          zip(summary[f"{PRIMARY_METRIC}_ci_low"], summary[f"{PRIMARY_METRIC}_ci_high"])]
save_table(main_table, "table_main_results", "Effectiveness of all systems (mean over queries)")
save_table(significance[["system", "vs", "mean_diff", "diff_ci_low", "diff_ci_high", "cohens_dz", "wins", "ties", "losses",
                         "p_randomization", "p_randomization_holm"]], "table_significance",
           f"Paired comparisons on {PRIMARY_METRIC} (randomisation test, Holm-corrected)")
save_table(by_type.reset_index(), "table_by_query_type", f"{PRIMARY_METRIC} by query type")
save_table(latency, "table_latency", "Query latency (ms)")
if not surf_calls.empty:
    save_table(surf_metrics, "table_surfability", "Surfability calls vs. buoy observations")
print("Tables:", sorted(p.name for p in TABLE_DIR.glob("*.md")))
print("Figures:", sorted(p.name for p in FIG_DIR.glob("*.png")))
display(Markdown((TABLE_DIR / "table_main_results.md").read_text(encoding="utf-8")))